# Leakage-safe preprocessing

This notebook fits preprocessing on training data only. It creates a scaled pipeline for future Logistic Regression and an unscaled pipeline for future Random Forest or HistGradientBoosting models; it does not train a classifier.

### 1. Define paths and fingerprint protected inputs

**What the cell does:** Imports preprocessing tools, defines repository-relative paths, and records checksums for all three split files and the final split summary.  
**Why it is necessary:** Preprocessing must be reproducible and must never modify source splits.  
**What to understand:** The printed fingerprints identify the exact inputs and will be compared again after all transformations are saved.

In [1]:
from pathlib import Path
import hashlib

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
TRAIN_PATH = PROJECT_ROOT / "data" / "modeling" / "train.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "modeling" / "validation.csv"
TEST_PATH = PROJECT_ROOT / "data" / "modeling" / "test.csv"
FINAL_SPLIT_SUMMARY_PATH = PROJECT_ROOT / "reports" / "final_split_summary.csv"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "modeling" / "preprocessed"
MODEL_DIR = PROJECT_ROOT / "models" / "preprocessing"
SUMMARY_PATH = PROJECT_ROOT / "reports" / "preprocessing_summary.csv"
FEATURE_REPORT_PATH = PROJECT_ROOT / "reports" / "preprocessing_feature_report.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_paths = [TRAIN_PATH, VALIDATION_PATH, TEST_PATH, FINAL_SPLIT_SUMMARY_PATH]
missing_inputs = [str(path) for path in input_paths if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(f"Required inputs are missing: {missing_inputs}")
input_hashes_before = {path: sha256_file(path) for path in input_paths}
for path, fingerprint in input_hashes_before.items():
    print(f"{path}: {fingerprint}")

/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/train.csv: 0dc81724c22630f5a000acbfc23d785af15cdc613ba19c6d9b77576b8d1973a7
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/validation.csv: b0ad6bd60b33aad5581bdfb3b02b77db78e4991f9ca69a46994f9950e51031e4
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/test.csv: f55a2e30b25bf72cce3900c8481ea8826a1e9da736c0f0948af2fb696c92c3b1
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/final_split_summary.csv: 721bbd76a02866a2c72c4ae11e26ac6f3d70bf0fadf18a1718a8838d42eade0f


### 2. Load and validate split integrity

**What the cell does:** Loads train, validation, test, and the split summary; verifies schemas, target values, report row counts, and run/row separation.  
**Why it is necessary:** Preprocessing an overlapping or inconsistent split would preserve leakage into every later model.  
**What to understand:** All three source tables have identical columns, disjoint complete runs and rows, correct binary targets, and the expected sizes.

In [2]:
split_data = {
    "train": pd.read_csv(TRAIN_PATH),
    "validation": pd.read_csv(VALIDATION_PATH),
    "test": pd.read_csv(TEST_PATH),
}
final_split_summary = pd.read_csv(FINAL_SPLIT_SUMMARY_PATH)
target_column = "slowdown_in_5min"
reference_columns = split_data["train"].columns.tolist()

assert target_column in reference_columns
assert all(frame.columns.tolist() == reference_columns for frame in split_data.values())
assert all(set(frame[target_column].dropna().unique()) <= {0, 1} for frame in split_data.values())
assert all(frame[target_column].notna().all() for frame in split_data.values())

run_sets = {name: set(frame["run_id"]) for name, frame in split_data.items()}
assert run_sets["train"].isdisjoint(run_sets["validation"])
assert run_sets["train"].isdisjoint(run_sets["test"])
assert run_sets["validation"].isdisjoint(run_sets["test"])

row_identity = ["machine_id", "run_id", "segment_id", "timestamp"]
row_sets = {
    name: set(map(tuple, frame[row_identity].itertuples(index=False, name=None)))
    for name, frame in split_data.items()
}
assert row_sets["train"].isdisjoint(row_sets["validation"])
assert row_sets["train"].isdisjoint(row_sets["test"])
assert row_sets["validation"].isdisjoint(row_sets["test"])

expected_rows = final_split_summary.set_index("split")["total_rows"].astype(int).to_dict()
assert all(len(frame) == expected_rows[name] for name, frame in split_data.items())

print("Validated split shapes:")
for name, frame in split_data.items():
    print(f"{name}: {frame.shape}, runs={frame['run_id'].nunique()}, positive rate={frame[target_column].mean() * 100:.4f}%")

Validated split shapes:
train: (14993, 251), runs=5, positive rate=7.2167%
validation: (1872, 251), runs=2, positive rate=37.9274%
test: (9171, 251), runs=3, positive rate=20.1832%


### 3. Declare metadata, prohibited fields, and numeric candidates

**What the cell does:** Defines identifier/target metadata, leakage exclusions, Rule C name patterns, and the candidate feature list without consulting target correlations.  
**Why it is necessary:** Explicit exclusions prevent identifiers, outcomes, label-definition intermediates, and future-comparison labels from entering model inputs.  
**What to understand:** Candidate features are everything not explicitly excluded; training-only auditing decides only whether a candidate is structurally usable.

In [3]:
identifier_metadata_columns = [column for column in [
    "id", "machine_id", "run_id", "segment_id", "timestamp",
    "valid_5min_horizon", "valid_10min_horizon", "slowdown_in_10min"
] if column in reference_columns]

explicit_exclusions = {
    "id", "machine_id", "run_id", "segment_id", "timestamp",
    "slowdown_in_5min", "slowdown_in_10min", "valid_5min_horizon",
    "valid_10min_horizon", "slowdown_now", "status", "ended_at_utc",
    "run_complete", "phase", "legacy_id", "stress_cpu_target_pct",
    "stress_memory_target_mb", "missed_deadline", "sensor_errors_json",
    "sample_reliable", "temperature_c", "gpu_usage_pct",
    "cpu_per_core_json", "gpu_per_device_json"
}
rule_c_prefixes = ("moderate_", "severe_", "rule_a_", "rule_b_", "rule_c_")

def is_prohibited(column):
    return column in explicit_exclusions or column.startswith(rule_c_prefixes) or "slowdown" in column.lower()

candidate_feature_columns = [column for column in reference_columns if not is_prohibited(column)]
duplicated_feature_names = pd.Index(candidate_feature_columns)[pd.Index(candidate_feature_columns).duplicated()].tolist()
validation_only_columns = sorted(set(split_data["validation"].columns) - set(split_data["train"].columns))
test_only_columns = sorted(set(split_data["test"].columns) - set(split_data["train"].columns))

print(f"Identifier/metadata columns kept separately: {identifier_metadata_columns}")
print(f"Candidate feature columns before train-only audit: {len(candidate_feature_columns)}")
print(f"Duplicated candidate names: {duplicated_feature_names}")
print(f"Validation-only columns: {validation_only_columns}")
print(f"Test-only columns: {test_only_columns}")

Identifier/metadata columns kept separately: ['machine_id', 'run_id', 'segment_id', 'timestamp', 'valid_5min_horizon', 'valid_10min_horizon', 'slowdown_in_10min']
Candidate feature columns before train-only audit: 243
Duplicated candidate names: []
Validation-only columns: []
Test-only columns: []


### 4. Perform the feature audit using training data only

**What the cell does:** Measures training dtype, missingness, uniqueness, infinity, and median; removes only all-missing, constant, or non-numeric candidates and records a reason for every input column.  
**Why it is necessary:** Structural decisions must be learned from train without target-based selection or validation/test statistics.  
**What to understand:** High missingness is reported but retained; exact retained feature order becomes fixed for all three splits.

In [4]:
X_train_audit = split_data["train"][candidate_feature_columns]
non_numeric_columns = [
    column for column in candidate_feature_columns
    if not pd.api.types.is_numeric_dtype(X_train_audit[column])
]
completely_missing_columns = [column for column in candidate_feature_columns if X_train_audit[column].isna().all()]
constant_columns = [
    column for column in candidate_feature_columns
    if column not in completely_missing_columns and X_train_audit[column].nunique(dropna=True) <= 1
]
infinite_counts = {
    column: int(np.isinf(X_train_audit[column].to_numpy(dtype=float)).sum())
    for column in candidate_feature_columns if column not in non_numeric_columns
}
columns_with_infinite_values = [column for column, count in infinite_counts.items() if count > 0]
removed_structural_columns = set(completely_missing_columns + constant_columns + non_numeric_columns)
retained_feature_columns = [column for column in candidate_feature_columns if column not in removed_structural_columns]
training_missing_feature_columns = [column for column in retained_feature_columns if X_train_audit[column].isna().any()]
high_missingness_columns = [
    column for column in retained_feature_columns if X_train_audit[column].isna().mean() >= 0.20
]

feature_report_records = []
for column in reference_columns:
    series = split_data["train"][column]
    missing_count = int(series.isna().sum())
    if is_prohibited(column):
        retained = False
        reason = "excluded_identifier_target_metadata_or_leakage"
    elif column in non_numeric_columns:
        retained = False
        reason = "removed_invalid_non_numeric_training_feature"
    elif column in completely_missing_columns:
        retained = False
        reason = "removed_completely_missing_in_training"
    elif column in constant_columns:
        retained = False
        reason = "removed_constant_in_training"
    else:
        retained = True
        reason = "retained_numeric_training_feature"
    feature_report_records.append({
        "feature_name": column,
        "training_data_type": str(series.dtype),
        "training_missing_count": missing_count,
        "training_missing_percentage": round(missing_count / len(series) * 100, 4),
        "training_unique_count": int(series.nunique(dropna=True)),
        "training_infinite_count": infinite_counts.get(column, np.nan),
        "retained_or_removed": "retained" if retained else "removed",
        "reason": reason,
        "median_learned_from_train": float(series.median()) if retained else np.nan,
        "missing_indicator_created": bool(retained and missing_count > 0),
        "high_training_missingness_20pct": bool(retained and missing_count / len(series) >= 0.20),
    })
preprocessing_feature_report = pd.DataFrame(feature_report_records)

assert not duplicated_feature_names and not validation_only_columns and not test_only_columns
assert not columns_with_infinite_values, f"Infinite training values require review: {columns_with_infinite_values}"
print(f"Completely missing training columns: {completely_missing_columns}")
print(f"Constant training columns removed: {constant_columns}")
print(f"Non-numeric candidates removed: {non_numeric_columns}")
print(f"Columns containing infinity: {columns_with_infinite_values}")
print(f"Retained original features: {len(retained_feature_columns)}")
print(f"Train-missing features receiving indicators: {len(training_missing_feature_columns)}")
print(f"High-missingness retained features: {high_missingness_columns}")

Completely missing training columns: []
Constant training columns removed: ['cpu_pct_missing_pct_30s', 'cpu_pct_missing_pct_60s', 'cpu_pct_missing_pct_120s', 'ram_pct_missing_pct_30s', 'ram_pct_missing_pct_60s', 'ram_pct_missing_pct_120s', 'swap_pct_missing_pct_30s', 'swap_pct_missing_pct_60s', 'swap_pct_missing_pct_120s', 'process_count_missing_pct_30s', 'process_count_missing_pct_60s', 'process_count_missing_pct_120s', 'thread_count_missing_pct_30s', 'thread_count_missing_pct_60s', 'thread_count_missing_pct_120s']
Non-numeric candidates removed: []
Columns containing infinity: []
Retained original features: 228
Train-missing features receiving indicators: 126
High-missingness retained features: ['context_switches_per_s', 'context_switches_per_s_change_30s', 'context_switches_per_s_change_60s', 'context_switches_per_s_change_120s', 'context_switches_per_s_diff']


### 5. Create aligned feature, target, and identifier tables

**What the cell does:** Applies the train-derived retained feature list to every split and separates binary targets and identifier/metadata tables.  
**Why it is necessary:** Validation and test must use exactly the training schema, while identifiers remain available for later error analysis without entering models.  
**What to understand:** `X_*` contains only retained numeric predictors; `y_*` contains only the main target; identifiers and the 10-minute comparison label are separate.

In [5]:
X_train = split_data["train"][retained_feature_columns].copy()
X_validation = split_data["validation"][retained_feature_columns].copy()
X_test = split_data["test"][retained_feature_columns].copy()
y_train = split_data["train"][[target_column]].astype("int8").copy()
y_validation = split_data["validation"][[target_column]].astype("int8").copy()
y_test = split_data["test"][[target_column]].astype("int8").copy()
identifiers = {
    name: frame[identifier_metadata_columns].copy()
    for name, frame in split_data.items()
}

assert X_train.columns.tolist() == X_validation.columns.tolist() == X_test.columns.tolist()
assert all(pd.api.types.is_numeric_dtype(X_train[column]) for column in X_train.columns)
assert not (set(retained_feature_columns) & explicit_exclusions)
assert not any(is_prohibited(column) for column in retained_feature_columns)

for split_name, X_frame, y_frame in [
    ("train", X_train, y_train), ("validation", X_validation, y_validation), ("test", X_test, y_test)
]:
    print(f"{split_name}: X={X_frame.shape}, y={y_frame.shape}, identifiers={identifiers[split_name].shape}")

train: X=(14993, 228), y=(14993, 1), identifiers=(14993, 7)
validation: X=(1872, 228), y=(1872, 1), identifiers=(1872, 7)
test: X=(9171, 228), y=(9171, 1), identifiers=(9171, 7)


### 6. Fit linear and tree preprocessors on training data only

**What the cell does:** Fits median-plus-indicator imputers on `X_train`; the linear pipeline additionally fits `StandardScaler`. Validation and test receive `transform` calls only.  
**Why it is necessary:** Train-only medians and scaling statistics prevent validation/test distribution leakage. Tree models do not need standardized scales.  
**What to understand:** Both pipelines share the same imputation logic and output width; only the linear output is standardized.

In [6]:
linear_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
])

# These are the only fit calls in the notebook.
linear_train_array = linear_preprocessor.fit_transform(X_train)
tree_train_array = tree_preprocessor.fit_transform(X_train)
linear_validation_array = linear_preprocessor.transform(X_validation)
linear_test_array = linear_preprocessor.transform(X_test)
tree_validation_array = tree_preprocessor.transform(X_validation)
tree_test_array = tree_preprocessor.transform(X_test)

fit_provenance = {
    "linear_preprocessor_fit_source": "X_train_only",
    "tree_preprocessor_fit_source": "X_train_only",
    "validation_operation": "transform_only",
    "test_operation": "transform_only",
}
display(pd.Series(fit_provenance, name="provenance").to_frame())

,provenance
linear_preprocessor_fit_source,X_train_only
tree_preprocessor_fit_source,X_train_only
validation_operation,transform_only
test_operation,transform_only


### 7. Recover names and verify learned train-only statistics

**What the cell does:** Names original and generated indicator columns, reconstructs DataFrames, checks medians/scaler statistics against training calculations, and rejects any missing or infinite transformed value.  
**Why it is necessary:** Clear schemas support interpretation, while numerical assertions prove that validation/test were not used to learn preprocessing parameters.  
**What to understand:** Linear and tree tables have identical named schemas and order; their values differ only because scaling is applied to the linear version.

In [7]:
linear_imputer = linear_preprocessor.named_steps["imputer"]
tree_imputer = tree_preprocessor.named_steps["imputer"]
indicator_feature_indices = linear_imputer.indicator_.features_.astype(int).tolist()
indicator_source_features = [retained_feature_columns[index] for index in indicator_feature_indices]
transformed_feature_names = retained_feature_columns + [
    f"{feature_name}__missing_indicator" for feature_name in indicator_source_features
]

assert indicator_source_features == training_missing_feature_columns
assert tree_imputer.indicator_.features_.astype(int).tolist() == indicator_feature_indices
assert len(transformed_feature_names) == linear_train_array.shape[1] == tree_train_array.shape[1]
assert len(transformed_feature_names) == len(set(transformed_feature_names))

training_medians = X_train.median().to_numpy(dtype=float)
assert np.allclose(linear_imputer.statistics_, training_medians, equal_nan=True)
assert np.allclose(tree_imputer.statistics_, training_medians, equal_nan=True)
imputed_train_for_scaler = linear_imputer.transform(X_train)
linear_scaler = linear_preprocessor.named_steps["scaler"]
assert np.allclose(linear_scaler.mean_, imputed_train_for_scaler.mean(axis=0))
assert np.allclose(linear_scaler.var_, imputed_train_for_scaler.var(axis=0))

def transformed_frame(array, columns):
    frame = pd.DataFrame(array, columns=columns)
    assert not frame.isna().any().any()
    assert np.isfinite(frame.to_numpy(dtype=float)).all()
    return frame

linear_frames = {
    "train": transformed_frame(linear_train_array, transformed_feature_names),
    "validation": transformed_frame(linear_validation_array, transformed_feature_names),
    "test": transformed_frame(linear_test_array, transformed_feature_names),
}
tree_frames = {
    "train": transformed_frame(tree_train_array, transformed_feature_names),
    "validation": transformed_frame(tree_validation_array, transformed_feature_names),
    "test": transformed_frame(tree_test_array, transformed_feature_names),
}
assert all(frame.columns.tolist() == transformed_feature_names for frame in [*linear_frames.values(), *tree_frames.values()])
print(f"Retained original features: {len(retained_feature_columns)}")
print(f"Generated train-derived indicators: {len(indicator_source_features)}")
print(f"Final transformed features: {len(transformed_feature_names)}")

Retained original features: 228
Generated train-derived indicators: 126
Final transformed features: 354


### 8. Build preprocessing reports

**What the cell does:** Summarizes rows, candidate/retained/indicator counts, missingness before and after, and target balance for every split; it also attaches fitted train medians to the feature audit.  
**Why it is necessary:** Reports make structural removals, learned values, and class differences auditable before model training.  
**What to understand:** Missing values before preprocessing can differ by split, but after preprocessing all are zero because one train-fitted imputer was reused.

In [8]:
preprocessing_summary_records = []
for split_name, X_frame, y_frame in [
    ("train", X_train, y_train),
    ("validation", X_validation, y_validation),
    ("test", X_test, y_test),
]:
    positives = int(y_frame[target_column].eq(1).sum())
    negatives = int(y_frame[target_column].eq(0).sum())
    preprocessing_summary_records.append({
        "split": split_name,
        "original_rows": int(len(X_frame)),
        "input_candidate_features": int(len(candidate_feature_columns)),
        "retained_original_features": int(len(retained_feature_columns)),
        "generated_missing_indicators": int(len(indicator_source_features)),
        "final_transformed_feature_count": int(len(transformed_feature_names)),
        "missing_values_before_preprocessing": int(X_frame.isna().sum().sum()),
        "missing_values_after_preprocessing_linear": int(linear_frames[split_name].isna().sum().sum()),
        "missing_values_after_preprocessing_tree": int(tree_frames[split_name].isna().sum().sum()),
        "positive_rows": positives,
        "negative_rows": negatives,
        "positive_rate": round(positives / len(y_frame) * 100, 4),
    })
preprocessing_summary = pd.DataFrame(preprocessing_summary_records)
display(preprocessing_summary)
print("Removed features:")
display(preprocessing_feature_report.loc[
    preprocessing_feature_report["retained_or_removed"].eq("removed"),
    ["feature_name", "reason"]
])

,split,original_rows,input_candidate_features,retained_original_features,generated_missing_indicators,final_transformed_feature_count,missing_values_before_preprocessing,missing_values_after_preprocessing_linear,missing_values_after_preprocessing_tree,positive_rows,negative_rows,positive_rate
0,train,14993,243,228,126,354,42174,0,0,1082,13911,7.2167
1,validation,1872,243,228,126,354,1066,0,0,710,1162,37.9274
2,test,9171,243,228,126,354,84708,0,0,1851,7320,20.1832


Removed features:


,feature_name,reason
0,machine_id,excluded_identifier_target_metadata_or_leakage
1,run_id,excluded_identifier_target_metadata_or_leakage
2,segment_id,excluded_identifier_target_metadata_or_leakage
3,timestamp,excluded_identifier_target_metadata_or_leakage
29,cpu_pct_missing_pct_30s,removed_constant_in_training
35,cpu_pct_missing_pct_60s,removed_constant_in_training
41,cpu_pct_missing_pct_120s,removed_constant_in_training
47,ram_pct_missing_pct_30s,removed_constant_in_training
53,ram_pct_missing_pct_60s,removed_constant_in_training
59,ram_pct_missing_pct_120s,removed_constant_in_training


### 9. Save preprocessors, transformed tables, targets, identifiers, and reports

**What the cell does:** Serializes both fitted pipelines and writes all requested linear/tree features, targets, identifiers, and audit reports.  
**Why it is necessary:** Reusing fitted train-only artifacts prevents accidental refitting during later validation, testing, or deployment.  
**What to understand:** Saved feature CSVs contain transformed predictors only; targets and identifiers remain separate.

In [9]:
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(linear_preprocessor, MODEL_DIR / "linear_preprocessor.joblib")
joblib.dump(tree_preprocessor, MODEL_DIR / "tree_preprocessor.joblib")

for split_name in ["train", "validation", "test"]:
    linear_frames[split_name].to_csv(PREPROCESSED_DIR / f"X_{split_name}_linear.csv", index=False)
    tree_frames[split_name].to_csv(PREPROCESSED_DIR / f"X_{split_name}_tree.csv", index=False)
    {"train": y_train, "validation": y_validation, "test": y_test}[split_name].to_csv(
        PREPROCESSED_DIR / f"y_{split_name}.csv", index=False
    )
    identifiers[split_name].to_csv(PREPROCESSED_DIR / f"{split_name}_identifiers.csv", index=False)

preprocessing_summary.to_csv(SUMMARY_PATH, index=False)
preprocessing_feature_report.to_csv(FEATURE_REPORT_PATH, index=False)

print(f"Saved linear pipeline: {MODEL_DIR / 'linear_preprocessor.joblib'}")
print(f"Saved tree pipeline: {MODEL_DIR / 'tree_preprocessor.joblib'}")
print(f"Saved preprocessed tables under: {PREPROCESSED_DIR}")
print(f"Saved reports: {SUMMARY_PATH}, {FEATURE_REPORT_PATH}")

Saved linear pipeline: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/preprocessing/linear_preprocessor.joblib
Saved tree pipeline: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/preprocessing/tree_preprocessor.joblib
Saved preprocessed tables under: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/modeling/preprocessed
Saved reports: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/preprocessing_summary.csv, /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/preprocessing_feature_report.csv


### 10. Reload artifacts and perform final leakage/integrity validation

**What the cell does:** Reloads pipelines and representative outputs, verifies schemas and finite values, confirms prohibited columns are absent, and recomputes all input checksums.  
**Why it is necessary:** Final read-back tests prove that saved artifacts—not only in-memory objects—are ready for later model training without leakage.  
**What to understand:** Success means preprocessing is complete; class balancing, feature selection, tuning, and model fitting remain future train-only tasks.

In [10]:
reloaded_linear = joblib.load(MODEL_DIR / "linear_preprocessor.joblib")
reloaded_tree = joblib.load(MODEL_DIR / "tree_preprocessor.joblib")
assert np.allclose(reloaded_linear.named_steps["imputer"].statistics_, training_medians)
assert np.allclose(reloaded_tree.named_steps["imputer"].statistics_, training_medians)

for split_name in ["train", "validation", "test"]:
    saved_linear = pd.read_csv(PREPROCESSED_DIR / f"X_{split_name}_linear.csv")
    saved_tree = pd.read_csv(PREPROCESSED_DIR / f"X_{split_name}_tree.csv")
    saved_y = pd.read_csv(PREPROCESSED_DIR / f"y_{split_name}.csv")
    saved_identifiers = pd.read_csv(PREPROCESSED_DIR / f"{split_name}_identifiers.csv")
    assert saved_linear.columns.tolist() == transformed_feature_names
    assert saved_tree.columns.tolist() == transformed_feature_names
    assert len(saved_linear) == len(saved_tree) == len(saved_y) == len(saved_identifiers) == len(split_data[split_name])
    assert not saved_linear.isna().any().any() and not saved_tree.isna().any().any()
    assert np.isfinite(saved_linear.to_numpy(dtype=float)).all()
    assert np.isfinite(saved_tree.to_numpy(dtype=float)).all()

assert not any(is_prohibited(column) for column in transformed_feature_names)
assert target_column not in transformed_feature_names
assert not any(column.startswith(rule_c_prefixes) for column in transformed_feature_names)
input_hashes_after = {path: sha256_file(path) for path in input_paths}
inputs_unchanged = input_hashes_before == input_hashes_after
assert inputs_unchanged, "An input split or report changed during preprocessing."

print("FINAL PREPROCESSING REPORT")
print(f"Input candidates: {len(candidate_feature_columns)}")
print(f"Retained original numeric features: {len(retained_feature_columns)}")
print(f"Removed constants: {len(constant_columns)}; all-missing: {len(completely_missing_columns)}; non-numeric: {len(non_numeric_columns)}")
print(f"Generated train-derived missing indicators: {len(indicator_source_features)}")
print(f"Final transformed features: {len(transformed_feature_names)}")
print("Linear pipeline: train-median imputation + train-derived indicators + train-fitted StandardScaler.")
print("Tree pipeline: train-median imputation + train-derived indicators; no scaling.")
print("All saved transformed values are finite and non-missing with identical schemas.")
print(f"All input files remained unchanged: {inputs_unchanged}")
print("No class balancing, SMOTE, target-based selection, tuning, or model training was performed.")

FINAL PREPROCESSING REPORT
Input candidates: 243
Retained original numeric features: 228
Removed constants: 15; all-missing: 0; non-numeric: 0
Generated train-derived missing indicators: 126
Final transformed features: 354
Linear pipeline: train-median imputation + train-derived indicators + train-fitted StandardScaler.
Tree pipeline: train-median imputation + train-derived indicators; no scaling.
All saved transformed values are finite and non-missing with identical schemas.
All input files remained unchanged: True
No class balancing, SMOTE, target-based selection, tuning, or model training was performed.
